# Find-by-photo lab

Measures how well each way of matching a shop photo to the catalogue works,
then trains a matcher on the catalogue's own photos.

**Before running**
1. *Runtime → Change runtime type → T4 GPU.*
2. Click the key icon on the left (Secrets), add `STAFF_PASSWORD` with the
   site's staff password, and switch on *Notebook access*.
3. *Runtime → Run all.* The first run takes about an hour; photos and the
   trained model are kept in Google Drive (`MyDrive/saree_lab`).

**What gets compared**
- **Gemini (live site):** the matcher the site uses today.
- **DINOv2:** an open image model looking at small pieces of fabric, untrained.
- **DINOv2 trained:** the same, after practising on disguised catalogue photos.

**The test photos.** There's no set of real shop photos yet, so each test photo
is a piece of a catalogue photo disguised as a shop photo: cut small, turned,
creased like a fold, relit, blurred and phone-compressed. The part of the
catalogue photo it came from is hidden from DINOv2 during the search, so it
has to recognise the design from other parts of the saree. Gemini can't be
told to hide it, which makes its scores look a little better than they are.
Returns that staff confirmed on the site are real shop photos and are scored
separately.

In [ ]:
API = 'https://ravisareeemporium-iwcl.onrender.com'
MODEL_NAME = 'facebook/dinov2-base'
SEGMENT_MODEL = 'mattmdjaga/segformer_b2_clothes'
KEEP_ON_DRIVE = True        # keep photos and the trained model between sessions
QUERY_PHOTOS = 400          # catalogue photos turned into test photos (one spread, one folded each)
GEMINI_QUERIES = 150        # test photos also sent to the live site; each uses one Gemini search
UNSEEN_SHARE = 0.2          # designs kept out of training, to check the training generalises
EPOCHS = 8
STEPS_PER_EPOCH = 150
DESIGNS_PER_BATCH = 32
PIECES_PER_DESIGN = 4
TRAIN_ON_ALL_AT_END = True  # after measuring, train once more on every design and save that model
SEED = 7

# Local testing only
DATASET_FILE = None         # a saved dataset JSON instead of asking the live site
SMOKE = False               # tiny run that finishes on a laptop CPU

In [ ]:
import os, io, json, time, math, hashlib
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

try:
    from google.colab import userdata, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=False)

import numpy as np
import requests
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from transformers import AutoModel, AutoImageProcessor, SegformerForSemanticSegmentation

if os.environ.get('SAREE_LAB_SMOKE'):
    SMOKE = True
    DATASET_FILE = os.environ.get('SAREE_LAB_DATASET', DATASET_FILE)
if SMOKE:
    MODEL_NAME = 'facebook/dinov2-small'
    QUERY_PHOTOS, GEMINI_QUERIES = 12, 0
    EPOCHS, STEPS_PER_EPOCH, DESIGNS_PER_BATCH, PIECES_PER_DESIGN = 1, 3, 4, 2
    UNSEEN_SHARE = 0.3

if IN_COLAB and KEEP_ON_DRIVE:
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/saree_lab'
elif IN_COLAB:
    WORK = '/content/saree_lab'
else:
    WORK = os.path.abspath(os.environ.get('SAREE_LAB_DIR', 'saree_lab'))
for sub in ('photos', 'masks', 'returns', 'model'):
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)


def secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)


STAFF_PASSWORD = secret('STAFF_PASSWORD')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = 2 if IN_COLAB else 0
if device == 'cpu' and not SMOKE:
    print('No GPU found. Runtime → Change runtime type → T4 GPU, then run again.')
print('Working folder:', WORK, '| device:', device)

## 1. Download the catalogue
Asks the site for every catalogue photo and every confirmed return, then keeps a
copy of each, shrunk to 1600 pixels on the long side.

In [ ]:
PHOTO_SIDE = 1600
QUERY_SIDE = 1024


def fetch_dataset():
    if DATASET_FILE:
        with open(DATASET_FILE) as f:
            return json.load(f)
    if not STAFF_PASSWORD:
        raise SystemExit('Add STAFF_PASSWORD in Colab Secrets (key icon on the left) and switch on notebook access.')
    for attempt in range(3):
        try:
            # The free Render server can take a minute to wake up.
            r = requests.get(f'{API}/api/identify/dataset', headers={'x-admin-password': STAFF_PASSWORD}, timeout=180)
        except requests.RequestException as err:
            print('Site did not answer, trying again:', err)
            time.sleep(10)
            continue
        if r.status_code == 401:
            raise SystemExit('The site did not accept the staff password.')
        if r.status_code == 404:
            raise SystemExit("The site doesn't have the photo list yet; the server change needs to be deployed first.")
        r.raise_for_status()
        return r.json()
    raise SystemExit('Could not reach the site.')


def stable_name(url):
    return hashlib.sha1(urlparse(url).path.encode()).hexdigest()[:16]


def open_rgb(data):
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(data)))
    if img.mode in ('RGBA', 'LA', 'P'):
        img = img.convert('RGBA')
        white = Image.new('RGBA', img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(white, img)
    return img.convert('RGB')


def shrink(img, side=PHOTO_SIDE):
    img = img.copy()
    img.thumbnail((side, side), Image.LANCZOS)
    return img


def fetch_photo(url, path):
    if os.path.exists(path):
        return True
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            shrink(open_rgb(r.content)).save(path + '.part', 'JPEG', quality=92)
            os.replace(path + '.part', path)
            return True
        except Exception as err:
            if attempt == 2:
                print('Skipped', url.split('?')[0][-60:], '-', err)
            time.sleep(2)
    return False


data = fetch_dataset()
for v in data['variants']:
    v['file'] = f"{WORK}/photos/{stable_name(v['image_path'])}.jpg"
for r in data['returns']:
    r['file'] = f"{WORK}/returns/{stable_name(r['photo_url'])}.jpg"

with ThreadPoolExecutor(8) as pool:
    ok = list(pool.map(lambda v: fetch_photo(v['image_path'], v['file']), data['variants']))
    ok_returns = list(pool.map(lambda r: fetch_photo(r['photo_url'], r['file']), data['returns']))

variants = [v for v, good in zip(data['variants'], ok) if good]
returns = [r for r, good in zip(data['returns'], ok_returns) if good]
designs = sorted({v['product_id'] for v in variants})
design_index = {d: i for i, d in enumerate(designs)}
variant_design = np.array([design_index[v['product_id']] for v in variants])
variants_of_design = [np.flatnonzero(variant_design == d) for d in range(len(designs))]
returns = [r for r in returns if r['product_id'] in design_index]
print(f'{len(variants)} catalogue photos of {len(designs)} designs, {len(returns)} confirmed returns')

## 2. Find the saree in each catalogue photo
Catalogue photos show a model in a setting; only the saree matters. A clothing
segmentation model marks the saree, so the background, face and arms are ignored,
and each photo is cut down to the saree to save memory.

In [ ]:
MASK_SCALE = 4              # masks are stored at a quarter of the photo size
CLOTH_NAMES = {'upper-clothes', 'skirt', 'pants', 'dress', 'belt', 'scarf'}

seg_processor = AutoImageProcessor.from_pretrained(SEGMENT_MODEL)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEGMENT_MODEL).to(device).eval()
cloth_labels = [i for i, name in seg_model.config.id2label.items() if name.lower() in CLOTH_NAMES]


@torch.no_grad()
def saree_mask(img):
    inputs = seg_processor(images=img, return_tensors='pt').to(device)
    logits = seg_model(**inputs).logits
    size = (max(img.height // MASK_SCALE, 1), max(img.width // MASK_SCALE, 1))
    labels = F.interpolate(logits, size=size, mode='bilinear', align_corners=False).argmax(1)[0].cpu().numpy()
    mask = np.isin(labels, cloth_labels).astype(np.uint8)
    mask = cv2.erode(mask, np.ones((3, 3), np.uint8), iterations=1)
    # A flat photo of fabric has no person in it, so the whole photo counts.
    if mask.mean() < 0.03:
        mask[:] = 1
    return mask


def cut_to_saree(arr, mask, margin=2):
    ys, xs = np.nonzero(mask)
    my0, my1 = max(ys.min() - margin, 0), min(ys.max() + margin + 1, mask.shape[0])
    mx0, mx1 = max(xs.min() - margin, 0), min(xs.max() + margin + 1, mask.shape[1])
    return (np.ascontiguousarray(arr[my0 * MASK_SCALE:my1 * MASK_SCALE, mx0 * MASK_SCALE:mx1 * MASK_SCALE]),
            np.ascontiguousarray(mask[my0:my1, mx0:mx1]))


photos, masks = [], []
for v in variants:
    img = Image.open(v['file']).convert('RGB')
    mask_file = v['file'].replace('/photos/', '/masks/').replace('.jpg', '.png')
    if os.path.exists(mask_file):
        mask = (np.asarray(Image.open(mask_file)) > 0).astype(np.uint8)
    else:
        mask = saree_mask(img)
        Image.fromarray(mask * 255).save(mask_file)
    photo, mask = cut_to_saree(np.asarray(img), mask)
    photos.append(photo)
    masks.append(mask)
mask_tables = [cv2.integral(m) for m in masks]
del seg_model
print(f'An average saree is {np.mean([p.shape[1] for p in photos]):.0f} x {np.mean([p.shape[0] for p in photos]):.0f} pixels; '
      f'photos use {sum(p.nbytes for p in photos) / 1e9:.1f} GB of memory')

## 3. Cut each saree into pieces, and make the disguised test photos

In [ ]:
INDEX_TILE_SIZES = (112, 176, 272, 400)  # pixels, on a photo 1600 pixels long
TILES_PER_SIZE = 6
MIN_COVER = 0.8                          # share of a piece that has to be saree
# piece size as a share of the shop photo's short side: how many pieces
QUERY_TILE_FRACTIONS = {1.0: 3, 0.6: 8, 0.4: 13}
QUICK_TILE_FRACTIONS = {1.0: 1, 0.6: 4, 0.4: 6}
COLOUR_BINS = 16
ROTATE_PAD = 1.42                        # room to turn a square piece without blank corners


def cover(table, box):
    x0, y0, x1, y1 = (int(c) // MASK_SCALE for c in box)
    area = max((x1 - x0) * (y1 - y0), 1)
    return (table[y1, x1] - table[y0, x1] - table[y1, x0] + table[y0, x0]) / area


def spread_pick(items, count):
    if len(items) <= count:
        return items
    return [items[i] for i in np.linspace(0, len(items) - 1, count).round().astype(int)]


def index_boxes(i):
    h, w = photos[i].shape[:2]
    table = mask_tables[i]
    boxes = []
    for s in INDEX_TILE_SIZES:
        if s > min(w, h):
            continue
        step = s // 2
        fits = [(x, y, x + s, y + s) for y in range(0, h - s + 1, step) for x in range(0, w - s + 1, step)
                if cover(table, (x, y, x + s, y + s)) >= MIN_COVER]
        boxes += spread_pick(fits, TILES_PER_SIZE)
    if not boxes:
        ys, xs = np.nonzero(masks[i])
        cx, cy = int(xs.mean()) * MASK_SCALE, int(ys.mean()) * MASK_SCALE
        s = min(w, h) // 2
        x, y = min(max(cx - s // 2, 0), w - s), min(max(cy - s // 2, 0), h - s)
        boxes = [(x, y, x + s, y + s)]
    return boxes


def colour_hist(rgb):
    small = cv2.resize(rgb, (64, 64), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).reshape(-1, 3)
    keep = (lab[:, 0] > 25) & (lab[:, 0] < 240)   # colour can't be told in deep shadow or glare
    if keep.sum() < 20:
        keep[:] = True
    hist, _, _ = np.histogram2d(lab[keep, 1], lab[keep, 2], bins=COLOUR_BINS, range=[[0, 256], [0, 256]])
    hist = cv2.GaussianBlur(hist.astype(np.float32), (3, 3), 0)
    return (hist / hist.sum()).ravel()


tile_box, tile_variant, tile_colour = [], [], []
for i in range(len(variants)):
    for box in index_boxes(i):
        x0, y0, x1, y1 = box
        tile_box.append(box)
        tile_variant.append(i)
        tile_colour.append(colour_hist(photos[i][y0:y1, x0:x1]))
tile_box = np.array(tile_box)
tile_variant = np.array(tile_variant)
tile_colour = np.array(tile_colour, dtype=np.float32)
print(f'{len(tile_box)} catalogue pieces, {len(tile_box) / len(variants):.1f} per photo')


def sample_box(i, piece, rng, min_cover, tries=60):
    """A square around a piece of saree, with room to turn the piece. None if none fits."""
    h, w = photos[i].shape[:2]
    size = min(int(piece * ROTATE_PAD), w, h)
    piece = int(size / ROTATE_PAD)
    for _ in range(tries):
        x, y = int(rng.integers(0, w - size + 1)), int(rng.integers(0, h - size + 1))
        o = (size - piece) // 2
        # The piece itself has to be saree; the turning room may show some background.
        if (cover(mask_tables[i], (x + o, y + o, x + o + piece, y + o + piece)) >= min_cover
                and cover(mask_tables[i], (x, y, x + size, y + size)) >= min_cover - 0.25):
            return (x, y, x + size, y + size)
    return None


def fold(a, rng):
    h, w = a.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    dx, dy = np.zeros((h, w), np.float32), np.zeros((h, w), np.float32)
    shade = np.ones((h, w), np.float32)
    for _ in range(int(rng.integers(1, 4))):
        angle = rng.uniform(0, np.pi)
        nx, ny = np.cos(angle), np.sin(angle)
        px, py = rng.uniform(0.2, 0.8) * w, rng.uniform(0.2, 0.8) * h
        d = (xx - px) * nx + (yy - py) * ny
        width = rng.uniform(0.02, 0.08) * max(h, w)
        shade *= 1 - rng.uniform(0.15, 0.4) * np.exp(-(d / width) ** 2)
        shade *= np.where(d > 0, rng.uniform(0.8, 1.05), 1.0).astype(np.float32)
        bunch = rng.uniform(0.5, 2.0) * width * np.exp(-(d / (2 * width)) ** 2)
        dx += bunch * nx
        dy += bunch * ny
    a = cv2.remap(a, xx + dx, yy + dy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return a * shade[..., None]


def relight(a, rng):
    h, w = a.shape[:2]
    warmth = rng.uniform(-1, 1)   # warm shop bulbs to cool tube lights
    a = a * (np.array([1 + 0.12 * warmth, 1, 1 - 0.12 * warmth]) * rng.uniform(0.92, 1.08, 3)).astype(np.float32)
    angle = rng.uniform(0, 2 * np.pi)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    ramp = ((xx / w - 0.5) * np.cos(angle) + (yy / h - 0.5) * np.sin(angle)) * 2
    a = a * (1 + rng.uniform(0, 0.25) * ramp)[..., None]
    a = (a - 0.5) * rng.uniform(0.75, 1.25) + 0.5 + rng.uniform(-0.1, 0.1)
    if rng.random() < 0.3:
        cx, cy, r = rng.uniform(0, w), rng.uniform(0, h), rng.uniform(0.15, 0.5) * max(h, w)
        a = a + rng.uniform(0.05, 0.2) * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r * r))[..., None]
    return a


def perspective(a, rng, strength):
    h, w = a.shape[:2]
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = src + rng.uniform(-strength, strength, (4, 2)).astype(np.float32) * np.float32([w, h])
    return cv2.warpPerspective(a, cv2.getPerspectiveTransform(src, dst), (w, h), borderMode=cv2.BORDER_REFLECT)


def disguise(i, box, kind, rng, out_side):
    """A piece of catalogue photo i made to look like a shop photo. box has ROTATE_PAD room around the piece."""
    x0, y0, x1, y1 = box
    big = photos[i][y0:y1, x0:x1].astype(np.float32) / 255
    side = x1 - x0
    s = int(side / ROTATE_PAD)
    turn = cv2.getRotationMatrix2D((side / 2, side / 2), rng.uniform(0, 360), 1.0)
    big = cv2.warpAffine(big, turn, (side, side), borderMode=cv2.BORDER_REFLECT)
    o = (side - s) // 2
    a = np.ascontiguousarray(big[o:o + s, o:o + s])
    if rng.random() < 0.5:
        a = a[:, ::-1].copy()
    if kind == 'folded':
        a = fold(a, rng)
    a = perspective(a, rng, 0.08 if kind == 'spread' else 0.04)
    a = cv2.resize(a, (out_side, out_side), interpolation=cv2.INTER_CUBIC if out_side > s else cv2.INTER_AREA)
    a = relight(a, rng)
    a = a + rng.normal(0, rng.uniform(0, 0.02), a.shape).astype(np.float32)
    blur = rng.uniform(0, 1.2) * out_side / 512
    if blur > 0.3:
        a = cv2.GaussianBlur(a, (0, 0), blur)
    a = (np.clip(a, 0, 1) * 255).astype(np.uint8)
    if kind == 'spread' and rng.random() < 0.5:
        pad = [int(out_side * rng.uniform(0, 0.15)) for _ in range(4)]
        grey = int(rng.integers(70, 190))   # a dull counter or sheet
        table = [int(np.clip(grey + rng.integers(-20, 20), 0, 255)) for _ in range(3)]
        a = cv2.copyMakeBorder(a, *pad, cv2.BORDER_CONSTANT, value=table)
    ok, jpeg = cv2.imencode('.jpg', cv2.cvtColor(a, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(45, 90))])
    return jpeg.tobytes()


def overlapping_tiles(i, box, share=0.3):
    """Pieces of photo i that overlap box, so a test can hide where the test photo came from."""
    mine = np.flatnonzero(tile_variant == i)
    b = tile_box[mine]
    ix = np.clip(np.minimum(b[:, 2], box[2]) - np.maximum(b[:, 0], box[0]), 0, None)
    iy = np.clip(np.minimum(b[:, 3], box[3]) - np.maximum(b[:, 1], box[1]), 0, None)
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return mine[ix * iy > share * area]


rng = np.random.default_rng(SEED)
unseen_designs = set(rng.choice(len(designs), max(1, int(len(designs) * UNSEEN_SHARE)), replace=False).tolist())

queries = []
for i in rng.permutation(len(variants))[:QUERY_PHOTOS]:
    saree_width = photos[i].shape[1]
    # piece size as a share of the saree's width in the catalogue photo
    for kind, shares in (('spread', (0.45, 0.7)), ('folded', (0.2, 0.4))):
        box = None
        for shrink_by in (1.0, 0.8, 0.65):
            box = sample_box(i, int(saree_width * rng.uniform(*shares) * shrink_by), rng, 0.85)
            if box is not None:
                break
        if box is None:
            continue
        o = int((box[2] - box[0]) * (1 - 1 / ROTATE_PAD) / 2)
        hidden = overlapping_tiles(i, (box[0] + o, box[1] + o, box[2] - o, box[3] - o))
        if len(np.setdiff1d(np.flatnonzero(tile_variant == i), hidden)) < 3 and len(variants_of_design[variant_design[i]]) == 1:
            continue   # nothing else of this design left to find it by
        out_side = int(np.clip((box[2] - box[0]) / ROTATE_PAD * rng.uniform(1.5, 3.0), 448, 1100))
        d = int(variant_design[i])
        queries.append({'variant': int(i), 'design': d, 'kind': kind, 'hidden': hidden,
                        'split': 'unseen' if d in unseen_designs else 'seen',
                        'jpeg': disguise(i, box, kind, rng, out_side)})
print(f'{len(queries)} test photos made')

preview = [Image.open(io.BytesIO(q['jpeg'])).resize((200, 200)) for q in queries[:8]]
if preview:
    sheet = Image.new('RGB', (200 * len(preview), 200))
    for k, p in enumerate(preview):
        sheet.paste(p, (200 * k, 0))
    sheet.save(f'{WORK}/test_photo_examples.jpg')
    if IN_COLAB:
        from IPython.display import display
        display(sheet)

## 4. Matching with DINOv2
A shop photo is cut into pieces too (and turned four ways, since a folded saree
can lie any way round). For each catalogue photo, every shop piece finds its
closest catalogue piece; the better half of those scores is averaged. Colour is
compared separately and added with a small weight, so pattern decides most of it.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
COLOUR_WEIGHTS = (0.0, 0.05, 0.1, 0.2)


def to_tensor(pieces):
    arr = np.stack([cv2.resize(p, (224, 224), interpolation=cv2.INTER_AREA) for p in pieces]).astype(np.float32) / 255
    return torch.from_numpy(((arr - MEAN) / STD).transpose(0, 3, 1, 2).copy())


backbone = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()


def plain_dino(x):
    out = backbone(pixel_values=x)
    cls = F.normalize(out.pooler_output, dim=-1)
    patches = F.normalize(out.last_hidden_state[:, 1:].mean(1), dim=-1)
    return F.normalize(torch.cat([cls, patches], 1), dim=-1)


@torch.no_grad()
def embed(pieces, model_fn, batch=64):
    out = []
    for k in range(0, len(pieces), batch):
        x = to_tensor(pieces[k:k + batch]).to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device == 'cuda'):
            out.append(model_fn(x).float().cpu())
    return torch.cat(out)


def build_index(model_fn):
    vectors = []
    for start in range(0, len(tile_box), 512):
        pieces = [photos[tile_variant[k]][y0:y1, x0:x1] for k, (x0, y0, x1, y1) in
                  zip(range(start, start + 512), tile_box[start:start + 512])]
        vectors.append(embed(pieces, model_fn))
    return torch.cat(vectors).to(device)


def query_pieces(arr, turns, fractions):
    h, w = arr.shape[:2]
    short = min(w, h)
    pieces = []
    for frac, count in fractions.items():
        s = int(short * frac)
        step = max(s // 2, 1)
        boxes = [(x, y) for y in range(0, h - s + 1, step) for x in range(0, w - s + 1, step)]
        pieces += [arr[y:y + s, x:x + s] for x, y in spread_pick(boxes, count)]
    return [np.ascontiguousarray(np.rot90(p, t)) for t in range(turns) for p in pieces], len(pieces)


tile_variant_t = torch.from_numpy(tile_variant)
variant_design_t = torch.from_numpy(variant_design)
variant_colour = np.zeros((len(variants), tile_colour.shape[1]), np.float32)
np.add.at(variant_colour, tile_variant, tile_colour)
variant_colour /= np.maximum(np.bincount(tile_variant, minlength=len(variants)), 1)[:, None]


def describe(jpeg, model_fn, turns, fractions):
    arr = np.asarray(shrink(open_rgb(jpeg), QUERY_SIDE))
    pieces, per_turn = query_pieces(arr, turns, fractions)
    h, w = arr.shape[:2]
    centre = arr[h // 10: h - h // 10, w // 10: w - w // 10]
    return {'vectors': embed(pieces, model_fn).view(turns, per_turn, -1), 'colour': colour_hist(centre)}


def match(desc, index, hidden=None):
    """Pattern and colour similarity of a shop photo to every catalogue photo."""
    turns, per_turn, dim = desc['vectors'].shape
    sims = (desc['vectors'].reshape(-1, dim).to(index.device) @ index.T).float().cpu()
    colours = variant_colour
    if hidden is not None and len(hidden):
        sims[:, torch.from_numpy(hidden)] = -2
        i = tile_variant[hidden[0]]
        left = np.setdiff1d(np.flatnonzero(tile_variant == i), hidden)
        colours = variant_colour.copy()
        if len(left):
            colours[i] = tile_colour[left].mean(0)
    best = torch.full((sims.shape[0], len(variants)), -2.0)
    best.scatter_reduce_(1, tile_variant_t.expand_as(sims), sims, reduce='amax')
    best = best.view(turns, per_turn, -1)
    pattern = best.topk(max(1, per_turn // 2), dim=1).values.mean(1).max(0).values
    colour = torch.from_numpy(np.sqrt(colours * desc['colour']).sum(1))
    return pattern, colour


def rank_of(variant_scores, true_design):
    design_scores = torch.full((len(designs),), -9.0).scatter_reduce_(0, variant_design_t, variant_scores, reduce='amax')
    rank = int((design_scores > design_scores[true_design]).sum()) + 1
    top = int(design_scores.argmax())
    top_variant = int(variants_of_design[top][variant_scores[variants_of_design[top]].argmax()])
    return rank, top_variant


def run_dino(name, model_fn, turns=4, fractions=QUERY_TILE_FRACTIONS, index=None):
    if index is None:
        index = build_index(model_fn)
    started = time.time()
    matched = [match(describe(q['jpeg'], model_fn, turns, fractions), index, q['hidden']) for q in queries]
    per_photo = (time.time() - started) / max(len(queries), 1)
    by_weight = {}
    for w in COLOUR_WEIGHTS:
        rows = []
        for q, (pattern, colour) in zip(queries, matched):
            rank, top_variant = rank_of(pattern + w * colour, q['design'])
            rows.append({'kind': q['kind'], 'split': q['split'], 'rank': rank, 'variant_ok': top_variant == q['variant']})
        by_weight[w] = rows
    weight = max(COLOUR_WEIGHTS, key=lambda w: (sum(r['rank'] == 1 for r in by_weight[w]), -w))
    rows = by_weight[weight]
    for r in returns:
        with open(r['file'], 'rb') as f:
            pattern, colour = match(describe(f.read(), model_fn, turns, fractions), index)
        rank, _ = rank_of(pattern + weight * colour, design_index[r['product_id']])
        rows.append({'kind': 'real return', 'split': 'seen', 'rank': rank, 'variant_ok': None})
    print(f'{name}: best colour weight {weight}, {per_photo:.2f} s per photo on {device}')
    return {'name': name, 'rows': rows, 'colour_weight': weight, 'index': index,
            'turns': turns, 'fractions': fractions}


results = [run_dino('DINOv2', plain_dino)]

## 5. The current Gemini matcher on the live site
Sends some of the same test photos to the site's "Find by photo" search. Nothing
is saved on the site. Confirmed returns already record where Gemini ranked them.

In [ ]:
def run_gemini():
    rows = []
    if GEMINI_QUERIES and not DATASET_FILE:
        headers = {'x-admin-password': STAFF_PASSWORD}
        status = requests.get(f'{API}/api/identify/status', headers=headers, timeout=120).json()
        print(f"Gemini has indexed {status.get('indexed')} of {status.get('total')} catalogue photos")
        for q in queries[:GEMINI_QUERIES]:
            r = requests.post(f'{API}/api/identify', headers=headers, timeout=120,
                              files={'photo': ('photo.jpg', q['jpeg'], 'image/jpeg')})
            if r.status_code == 503:
                print('Stopped early:', r.json().get('error'))
                break
            r.raise_for_status()
            found = [m['product_id'] for m in r.json()['matches']]
            want = designs[q['design']]
            rank = found.index(want) + 1 if want in found else 99
            top_variant = None
            if found and found[0] == want:
                top_variant = next((i for i in variants_of_design[q['design']]
                                    if variants[i]['variant_number'] == r.json()['matches'][0]['variant_number']), None)
            rows.append({'kind': q['kind'], 'split': q['split'], 'rank': rank, 'variant_ok': top_variant == q['variant']})
    for r in returns:
        rows.append({'kind': 'real return', 'split': 'seen', 'rank': r['rank'] or 99, 'variant_ok': None})
    return {'name': 'Gemini (live site)', 'rows': rows}


results.append(run_gemini())


def report(results, split=None):
    print(f"\n{'':24}{'test photos':>14}{'photos':>8}{'right first':>13}{'in top 5':>10}{'right colour':>14}")
    for res in results:
        for kind in ('spread', 'folded', 'real return'):
            rows = [r for r in res['rows'] if r['kind'] == kind and (split is None or r['split'] == split)]
            if not rows:
                continue
            first = sum(r['rank'] == 1 for r in rows) / len(rows)
            top5 = sum(r['rank'] <= 5 for r in rows) / len(rows)
            colour_rows = [r for r in rows if r['rank'] == 1 and r['variant_ok'] is not None]
            colour = f"{sum(r['variant_ok'] for r in colour_rows) / len(colour_rows):.0%}" if colour_rows else '-'
            print(f"{res['name']:24}{kind:>14}{len(rows):>8}{first:>13.0%}{top5:>10.0%}{colour:>14}")


report(results)

## 6. Train DINOv2 on the catalogue
Practises on pairs of disguised pieces: pieces of the same design should end up
close together, pieces of different designs far apart, whatever the angle,
fold, light or blur. Colour variants of a design count as the same design, so it
learns pattern; colour is still compared separately when matching.
Designs kept aside (the "unseen" ones) show whether this helps with designs
added to the catalogue later.

In [ ]:
OUT_DIM = 384
TRAIN_BLOCKS = 4


class Matcher(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(2 * hidden, 1024), nn.GELU(), nn.Linear(1024, OUT_DIM))
        for p in self.backbone.parameters():
            p.requires_grad = False
        for part in [*self.backbone.encoder.layer[-TRAIN_BLOCKS:], self.backbone.layernorm]:
            for p in part.parameters():
                p.requires_grad = True

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        features = torch.cat([out.pooler_output, out.last_hidden_state[:, 1:].mean(1)], 1)
        return F.normalize(self.head(features), dim=-1)


class Pieces(torch.utils.data.Dataset):
    def __len__(self):
        return len(variants)

    def __getitem__(self, i):
        rng = np.random.default_rng()
        h, w = photos[i].shape[:2]
        biggest = max(int(min(h, w) / ROTATE_PAD), 113)
        s = int(np.exp(rng.uniform(np.log(112), np.log(min(560, biggest)))))
        box = sample_box(i, s, rng, MIN_COVER - 0.1)
        if box is None:
            box = index_boxes(i)[0]
        if rng.random() < 0.35:
            x0, y0, x1, y1 = box
            o = int((x1 - x0) * (1 - 1 / ROTATE_PAD) / 2)
            piece = np.rot90(photos[i][y0 + o:y1 - o, x0 + o:x1 - o], int(rng.integers(4)))
        else:
            piece = np.asarray(open_rgb(disguise(i, box, rng.choice(['spread', 'folded']), rng, 256)))
        return to_tensor([np.ascontiguousarray(piece)])[0], int(variant_design[i])


class DesignBatches(torch.utils.data.Sampler):
    def __init__(self, design_ids, steps):
        self.design_ids, self.steps = list(design_ids), steps

    def __len__(self):
        return self.steps

    def __iter__(self):
        rng = np.random.default_rng()
        for _ in range(self.steps):
            picked = rng.choice(self.design_ids, DESIGNS_PER_BATCH, replace=len(self.design_ids) < DESIGNS_PER_BATCH)
            yield [int(rng.choice(variants_of_design[d])) for d in picked for _ in range(PIECES_PER_DESIGN)]


def contrastive_loss(z, labels, temperature=0.07):
    z = z.float()
    sims = z @ z.T / temperature
    self_pairs = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sims = sims.masked_fill(self_pairs, -1e4)
    same = (labels[:, None] == labels[None, :]) & ~self_pairs
    log_prob = sims - torch.logsumexp(sims, dim=1, keepdim=True)
    return (-(log_prob * same).sum(1) / same.sum(1).clamp(min=1)).mean()


def train(design_ids):
    model = Matcher(MODEL_NAME).to(device)
    trainable = [p for p in model.backbone.parameters() if p.requires_grad]
    optimiser = torch.optim.AdamW([{'params': trainable, 'lr': 2e-5},
                                   {'params': model.head.parameters(), 'lr': 1e-3}], weight_decay=0.05)
    total = EPOCHS * STEPS_PER_EPOCH
    warmup = max(1, total // 20)
    schedule = torch.optim.lr_scheduler.LambdaLR(
        optimiser, lambda step: min(1, (step + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(step / total, 1))))
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')
    loader = torch.utils.data.DataLoader(Pieces(), batch_sampler=DesignBatches(design_ids, STEPS_PER_EPOCH),
                                         num_workers=NUM_WORKERS)
    model.train()
    for epoch in range(EPOCHS):
        started, losses = time.time(), []
        for x, labels in loader:
            x, labels = x.to(device), labels.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device == 'cuda'):
                loss = contrastive_loss(model(x), labels)
            optimiser.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimiser)
            scaler.update()
            schedule.step()
            losses.append(loss.item())
        print(f'Round {epoch + 1}/{EPOCHS}: loss {np.mean(losses):.3f} ({time.time() - started:.0f} s)')
    return model.eval()


seen_designs = [d for d in range(len(designs)) if d not in unseen_designs]
trained = train(seen_designs)
results.append(run_dino('DINOv2 trained', trained))
# Fewer pieces and no turning: what a free, slow server could afford per search.
results.append(run_dino('DINOv2 trained, quick', trained, turns=1, fractions=QUICK_TILE_FRACTIONS,
                        index=results[-1]['index']))

## 7. Scores
*right first*: the correct design was the top suggestion. *in top 5*: it was
among the five shown. *right colour*: when the design was right, the top
suggestion was also the right colour variant.
"Unseen" designs were never used in training, like designs added later.

In [ ]:
print('All test photos')
report(results)
for split in ('seen', 'unseen'):
    print(f'\n{split.capitalize()} designs only (test photos)')
    report([{**res, 'rows': [r for r in res['rows'] if r['kind'] != 'real return']} for res in results], split)

with open(f'{WORK}/scores.json', 'w') as f:
    json.dump([{'name': res['name'], 'colour_weight': res.get('colour_weight'), 'rows': res['rows']} for res in results], f)

## 8. Save the matcher
Trains once more on every design (if `TRAIN_ON_ALL_AT_END`) and saves it to
`saree_lab/model/saree_matcher.pt`, ready for the matching service.

In [ ]:
final = train(list(range(len(designs)))) if TRAIN_ON_ALL_AT_END else trained
trained_weight = next(res['colour_weight'] for res in results if res['name'] == 'DINOv2 trained')
torch.save({
    'model_name': MODEL_NAME,
    'out_dim': OUT_DIM,
    'train_blocks': TRAIN_BLOCKS,
    'state_dict': {k: v.half() for k, v in final.state_dict().items()},
    'settings': {
        'photo_side': PHOTO_SIDE, 'query_side': QUERY_SIDE, 'mask_scale': MASK_SCALE,
        'index_tile_sizes': INDEX_TILE_SIZES, 'tiles_per_size': TILES_PER_SIZE, 'min_cover': MIN_COVER,
        'query_tile_fractions': QUERY_TILE_FRACTIONS, 'quick_tile_fractions': QUICK_TILE_FRACTIONS,
        'colour_bins': COLOUR_BINS,
        'colour_weight': trained_weight, 'segment_model': SEGMENT_MODEL, 'cloth_names': sorted(CLOTH_NAMES),
    },
}, f'{WORK}/model/saree_matcher.pt')
print('Saved', f'{WORK}/model/saree_matcher.pt')